### Geoginah Ramatshe_26303686

# Predicting Retail Investor Dormancy on Digital Investment Platforms

**Project:** A Machine Learning Approach to Predicting Retail Investor Dormancy on Digital Investment Platforms  
**Models:** Logistic Regression and Gradient Boosting (XGBoost)

## Research design

This notebook focuses on **one prediction problem**: among customers who transacted in the 30 days before a monthly snapshot date, predict whether they will make no transaction during the following 90 days.

- `dormancy_target = 1`: no qualifying transaction during the next 90 days (dormant under this study's operational definition).
- `dormancy_target = 0`: at least one qualifying transaction during the next 90 days (remains active).

The target is derived from dated transaction records. The supplied `dormant_investor` and `active_investor` fields are **not used as predictors or as the target**. The supplied `days_since_last_activity` field is also excluded because its reference date is unclear and it may encode current inactivity directly.

> **Important limitation:** this is a transaction-based research definition. It should not be described as the client's official dormancy label unless the client confirms that the 90-day rule matches its policy.

## 1. Dataset loading

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Raw CSV files are expected to be in the same folder as these notebooks.
BASE_DIR = Path.cwd()
RAW_FILES = {
    "profile": "table1_customer_profile.csv",
    "funding": "table2_account_funding.csv",
    "transactions": "table3_transaction_trading.csv",
    "engagement": "table4_platform_engagement.csv",
}

def safe_date(series):
    return pd.to_datetime(series, errors="coerce")

missing_files = [name for name in RAW_FILES.values() if not (BASE_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Could not find these CSV files in the notebook's current folder: "
        + ", ".join(missing_files)
        + ". Place the CSV files beside the notebooks or update BASE_DIR.")

profile_raw = pd.read_csv(BASE_DIR / RAW_FILES["profile"])
funding_raw = pd.read_csv(BASE_DIR / RAW_FILES["funding"])
transaction_raw = pd.read_csv(BASE_DIR / RAW_FILES["transactions"])
engagement_raw = pd.read_csv(BASE_DIR / RAW_FILES["engagement"])

print("All four datasets loaded successfully.")
for name, frame in [
    ("Customer profile", profile_raw),
    ("Account funding", funding_raw),
    ("Transaction/trading", transaction_raw),
    ("Platform engagement", engagement_raw),
]:
    print(f"{name}: {frame.shape[0]:,} rows × {frame.shape[1]:,} columns")

All four datasets loaded successfully.
Customer profile: 5,433 rows × 9 columns
Account funding: 5,417 rows × 7 columns
Transaction/trading: 16,703 rows × 8 columns
Platform engagement: 57,529 rows × 5 columns


### Initial dataset size and quality

The raw tables contain repeated customer IDs by design in the transaction and engagement tables because a customer can have many events. The customer-profile table has exact duplicate rows, which will be removed. 
The transaction table contains 421 rows without usable transaction timestamps. These cannot be used to place activity before or after a snapshot date, so they will be excluded from the time-based target and behavioural features.

The modelling dataset is created at the **customer-month snapshot** level, not at the raw-event level. The class counts and imbalance proportion are reported again after the new dormancy target has been constructed.

In [2]:
raw_summary = pd.DataFrame([
    {
        "table": "Customer profile",
        "rows": len(profile_raw),
        "unique_customers": profile_raw["customer_id"].nunique(),
        "exact_duplicate_rows": int(profile_raw.duplicated().sum()),
        "missing_key_dates": int(safe_date(profile_raw["registration_date"]).isna().sum()),
    },
    {
        "table": "Account funding",
        "rows": len(funding_raw),
        "unique_customers": funding_raw["customer_id"].nunique(),
        "exact_duplicate_rows": int(funding_raw.duplicated().sum()),
        "missing_key_dates": int(safe_date(funding_raw["first_deposit_date"]).isna().sum()),
    },
    {
        "table": "Transaction/trading",
        "rows": len(transaction_raw),
        "unique_customers": transaction_raw["customer_id"].nunique(),
        "exact_duplicate_rows": int(transaction_raw.duplicated().sum()),
        "missing_key_dates": int(safe_date(transaction_raw["transaction_datetime"]).isna().sum()),
    },
    {
        "table": "Platform engagement",
        "rows": len(engagement_raw),
        "unique_customers": engagement_raw["customer_id"].nunique(),
        "exact_duplicate_rows": int(engagement_raw.duplicated().sum()),
        "missing_key_dates": int(safe_date(engagement_raw["login_date"]).isna().sum()),
    },
])
display(raw_summary)

,table,rows,unique_customers,exact_duplicate_rows,missing_key_dates
0,Customer profile,5433,5417,16,0
1,Account funding,5417,5417,0,2973
2,Transaction/trading,16703,2444,0,421
3,Platform engagement,57529,4617,0,0


## Date coverage

The earliest and latest valid dates help establish whether the data cover the historical snapshot dates and the 90-day future outcome window. A minimum and maximum date do not, by themselves, prove that every date in between is covered.

In [3]:
date_series = {
    "Registration date": safe_date(profile_raw["registration_date"]),
    "First deposit date": safe_date(funding_raw["first_deposit_date"]),
    "Transaction date": safe_date(transaction_raw["transaction_datetime"]),
    "Login date": safe_date(engagement_raw["login_date"]),
}
date_coverage = pd.DataFrame([
    {
        "field": name,
        "valid_records": int(values.notna().sum()),
        "first_date": values.min(),
        "last_date": values.max(),
    }
    for name, values in date_series.items()
])
display(date_coverage)

,field,valid_records,first_date,last_date
0,Registration date,5433,2024-06-01 01:18:00,2026-06-29 18:31:00
1,First deposit date,2444,2024-06-09 12:45:57,2026-08-03 22:28:51
2,Transaction date,16282,2024-06-23 18:15:32,2026-09-07 19:02:04
3,Login date,57529,2024-06-04 09:39:19,2026-09-07 23:32:28


## Dataset instances, target classes and imbalance

At this stage, the raw tables are event-level tables and do not yet have a single shared class label. The modelling instances will be **customer-month snapshots** created in the preprocessing notebook. Class counts and imbalance will be calculated there from the newly defined `dormancy_target`: `0 = remains active` and `1 = no transaction in the next 90 days` among customers eligible at the snapshot date. This avoids using the client's existing `active_investor` or `dormant_investor` columns as labels or predictors.